# 🚲 Bike Sharing Rental — DS Group 6

**Problem Statement:**  
Analyze the bike rental dataset to identify key factors affecting bike demand (weather, time, season) and build predictive regression models to forecast hourly rental demand.

**Models built:** Decision Tree · Random Forest · Gradient Boosting  
**Best model:** Gradient Boosting — R² 0.9519 · MAE 26.29 · RMSE 43.10

---

# `Exploratory Data Analysis (EDA)`

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import math

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
print("Libraries loaded successfully!")

## 2. Load Dataset

In [ ]:
df = pd.read_csv('Dataset.csv')
print(f"Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
print("Last 5 rows:")
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Data Cleaning

### 3.1 Handling Categorical Inconsistencies

In [ ]:
# Replace invalid entries and fix incorrect labels
df.replace('?', np.nan, inplace=True)

cat_cols = df.select_dtypes(include=['object', 'category']).columns
for col in cat_cols:
    df[col] = df[col].str.strip().str.lower()

df.replace(['nan', 'NaN', ' NAN'], np.nan, inplace=True)
df['season']     = df['season'].replace('springer', 'spring')
df['weathersit'] = df['weathersit'].replace({'lightrain': 'light rain'})
print("Categorical inconsistencies fixed!")

### 3.2 Data Type Correction

In [ ]:
df['dteday'] = pd.to_datetime(df['dteday'], dayfirst=True)

categorical_cols = ['season', 'yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit']
for col in categorical_cols:
    df[col] = df[col].astype('category')

num_cols = ['temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']
for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types corrected!")

### 3.3 Missing Values

In [ ]:
missing_values = df.isnull().sum()
print("Missing values per column:")
print(missing_values[missing_values > 0] if missing_values.any() else "  None found")

missing_pct = (df.isnull().sum().sum() / (df.shape[0] * df.shape[1])) * 100
print(f"\nTotal Missing: {df.isnull().sum().sum()} ({missing_pct:.4f}%)")
df = df.dropna()
print(f"Rows after dropping: {len(df):,}")

> **Note:** Missing values were less than 0.01% — safely dropped without impacting analysis.

### 3.4 Duplicates

In [ ]:
print(f"Duplicate rows: {df.duplicated().sum()}")

### 3.5 Remove Irrelevant Features

In [ ]:
if 'instant' in df.columns:
    df = df.drop('instant', axis=1)
    print("'instant' column dropped — it is a unique identifier with no predictive value.")
print("Final Shape:", df.shape)

### 3.6 Outlier Detection

In [ ]:
from sklearn.preprocessing import StandardScaler as _sc

_scaler = _sc()
_num = ['temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']
_scaled = pd.DataFrame(_scaler.fit_transform(df[_num]), columns=_num)

plt.figure(figsize=(14, 5))
sns.boxplot(data=_scaled)
plt.xticks(rotation=45)
plt.title("Outlier Detection — Normalised Features", fontweight='bold')
plt.tight_layout()
plt.show()

> **Outlier Strategy:** Outliers in `cnt`, `casual`, and `registered` represent genuine high-demand events (holidays, good weather). Retained intentionally — they carry valuable business signal.

## 4. Univariate Analysis

### 4.1 Numerical Features — Distribution

In [ ]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns

for col in num_cols:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[col], bins=50, kde=True, ax=axes[0])
    axes[0].set_title(f'Distribution of {col}')
    sns.boxplot(x=df[col], ax=axes[1])
    axes[1].set_title(f'Outliers in {col}')
    plt.tight_layout()
    plt.show()

### 4.2 Categorical Features

In [ ]:
cat_cols = ['yr', 'mnth', 'holiday', 'weekday', 'workingday', 'weathersit']
n_rows = math.ceil(len(cat_cols) / 3)
fig, axes = plt.subplots(n_rows, 3, figsize=(16, 4 * n_rows))
axes = axes.flatten()
palette = sns.color_palette("Set2", len(cat_cols))

for i, col in enumerate(cat_cols):
    sns.countplot(x=df[col], ax=axes[i], color=palette[i])
    axes[i].set_title(f'Distribution of {col}')

for j in range(len(cat_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

### 4.3 Target Variable — Bike Demand (`cnt`)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(df['cnt'], color='steelblue', linewidth=0.6)
axes[0].set_title('Bike Demand Over Time', fontweight='bold')
axes[0].set_xlabel('Index')
axes[0].set_ylabel('Count')

sns.lineplot(x=df['hr'], y=df['cnt'], ax=axes[1], color='orange')
axes[1].set_title('Average Bike Demand by Hour', fontweight='bold')

plt.tight_layout()
plt.show()

### 4.4 Season Distribution

In [ ]:
season_counts = df['season'].value_counts()
plt.figure(figsize=(6, 6))
plt.pie(season_counts, labels=season_counts.index, autopct='%1.1f%%',
        startangle=90, colors=sns.color_palette('Set2', len(season_counts)))
plt.title('Season Distribution', fontweight='bold')
plt.show()

## 5. Bivariate Analysis

### 5.1 Weather Features vs Demand

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.scatterplot(x=df['temp'],  y=df['cnt'], ax=axes[0], alpha=0.4, color='teal')
axes[0].set_title('Temperature vs Demand')

sns.scatterplot(x=df['atemp'], y=df['cnt'], ax=axes[1], alpha=0.4, color='orange')
axes[1].set_title('Feels-Like Temp vs Demand')

sns.scatterplot(x=df['hum'],   y=df['cnt'], ax=axes[2], alpha=0.4, color='purple')
axes[2].set_title('Humidity vs Demand')

plt.tight_layout()
plt.show()

### 5.2 Windspeed vs Demand

In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(x=df['windspeed'], y=df['cnt'], alpha=0.4, color='red')
plt.title('Windspeed vs Bike Demand', fontweight='bold')
plt.show()

### 5.3 Categorical Factors vs Demand

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.boxplot(x=df['workingday'], y=df['cnt'], palette='pastel', ax=axes[0])
axes[0].set_title('Working Day vs Demand')
axes[0].set_xticklabels(['Non-Working', 'Working Day'])

sns.boxplot(x=df['weathersit'], y=df['cnt'], palette='coolwarm', ax=axes[1])
axes[1].set_title('Weather vs Demand')

sns.boxplot(x=df['season'], y=df['cnt'], palette='Set2', ax=axes[2])
axes[2].set_title('Season vs Demand')

plt.tight_layout()
plt.show()

### 5.4 Temporal Trends

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

df['mnth'] = pd.to_numeric(df['mnth'])
monthly_avg = df.groupby('mnth')['cnt'].mean()
sns.lineplot(x=monthly_avg.index, y=monthly_avg.values,
             marker='o', color='green', ax=axes[0])
axes[0].set_title('Monthly Bike Demand Trend', fontweight='bold')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Average Demand')
axes[0].set_xticks(range(1, 13))
axes[0].set_xticklabels(['Jan','Feb','Mar','Apr','May','Jun',
                          'Jul','Aug','Sep','Oct','Nov','Dec'])

hourly_avg = df.groupby('hr')['cnt'].mean()
sns.lineplot(x=hourly_avg.index, y=hourly_avg.values,
             marker='o', ax=axes[1], color='steelblue')
axes[1].set_title('Hourly Demand Trend', fontweight='bold')
axes[1].set_xlabel('Hour')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

### 5.5 User Type Contribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.scatterplot(x=df['casual'],     y=df['cnt'], ax=axes[0], alpha=0.4, color='blue')
axes[0].set_title('Casual Users vs Total Demand')

sns.scatterplot(x=df['registered'], y=df['cnt'], ax=axes[1], alpha=0.4, color='green')
axes[1].set_title('Registered Users vs Total Demand')

plt.tight_layout()
plt.show()

> **Insight:** Registered users form the revenue backbone. Casual users drive weekend and holiday demand. Different strategies needed for each segment.

### 5.6 Correlation Matrix

In [ ]:
num_df = df.select_dtypes(include=['int64', 'float64'])

plt.figure(figsize=(10, 8))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f',
            cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Advanced EDA

### 6.1 Weekday × Hour Heatmap
> Reveals the two-peak commuter pattern on weekdays vs the leisure curve on weekends.

In [ ]:
pivot = df.groupby(['weekday', 'hr'])['cnt'].mean().unstack()
pivot.index = ['Sun','Mon','Tue','Wed','Thu','Fri','Sat']

plt.figure(figsize=(16, 5))
sns.heatmap(pivot, cmap='YlOrRd', linewidths=0.3, linecolor='white', annot=False)
plt.title('Average Bike Demand — Weekday × Hour', fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Day of Week')
plt.tight_layout()
plt.show()

### 6.2 Casual vs Registered Users by Hour
> Registered users peak at commute hours; casual users peak midday — two distinct behavioural patterns.

In [ ]:
hourly_users = df.groupby('hr')[['casual', 'registered']].mean()

plt.figure(figsize=(14, 5))
plt.plot(hourly_users.index, hourly_users['casual'],
         marker='o', label='Casual', color='#3498db', linewidth=2)
plt.plot(hourly_users.index, hourly_users['registered'],
         marker='s', label='Registered', color='#e74c3c', linewidth=2)
plt.fill_between(hourly_users.index, hourly_users['casual'],     alpha=0.15, color='#3498db')
plt.fill_between(hourly_users.index, hourly_users['registered'], alpha=0.15, color='#e74c3c')
plt.title('Casual vs Registered Users by Hour', fontweight='bold')
plt.xlabel('Hour of Day')
plt.ylabel('Average Rentals')
plt.xticks(range(0, 24))
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 6.3 Temperature Binned vs Demand
> Demand rises steadily with temperature — warm conditions drive peak rentals.

In [ ]:
df['temp_bin'] = pd.cut(df['temp'], bins=5,
                         labels=['Very Cold', 'Cold', 'Mild', 'Warm', 'Hot'])

plt.figure(figsize=(10, 5))
sns.boxplot(x='temp_bin', y='cnt', data=df, palette='coolwarm',
            order=['Very Cold', 'Cold', 'Mild', 'Warm', 'Hot'])
plt.title('Bike Demand by Temperature Range', fontweight='bold')
plt.xlabel('Temperature Range')
plt.ylabel('Rental Count')
plt.tight_layout()
plt.show()

### 6.4 Weather Condition — Violin Plot
> Clear weather shows the widest and highest demand distribution. Heavy rain reduces rentals to near zero.

In [ ]:
plt.figure(figsize=(10, 5))
sns.violinplot(x='weathersit', y='cnt', data=df, palette='Set2')
plt.title('Bike Demand Distribution by Weather Condition', fontweight='bold')
plt.xlabel('Weather Condition')
plt.ylabel('Rental Count')
plt.tight_layout()
plt.show()

### 6.5 Monthly Demand — 2011 vs 2012
> Demand grew significantly year-over-year, confirming system adoption and expansion.

In [ ]:
df['yr_num'] = df['yr'].astype(str)

monthly = df.groupby(['yr_num', 'mnth'])['cnt'].mean().reset_index()
monthly['mnth'] = monthly['mnth'].astype(int)

plt.figure(figsize=(14, 5))
for year, grp in monthly.groupby('yr_num'):
    plt.plot(grp['mnth'], grp['cnt'], marker='o', linewidth=2, label=year)

plt.title('Monthly Demand — 2011 vs 2012', fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Average Rentals')
plt.xticks(range(1, 13), ['Jan','Feb','Mar','Apr','May','Jun',
                            'Jul','Aug','Sep','Oct','Nov','Dec'])
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Feature Engineering

In [ ]:
# Time-based features
df['time_of_day'] = pd.cut(df['hr'],
    bins=[-1, 6, 12, 18, 24],
    labels=['Night', 'Morning', 'Afternoon', 'Evening'])

df['is_weekend'] = df['weekday'].isin([0, 6]).astype(int)

# Weather simplification
df['weather_group'] = df['weathersit'].replace({
    1: 'Clear',
    2: 'Mist/Cloudy',
    3: 'Light Rain/Snow',
    4: 'Heavy Rain/Snow',
})

# Temperature perception gap
df['temp_diff'] = df['atemp'] - df['temp']

print(f"New features added. Final shape: {df.shape}")
df[['time_of_day', 'is_weekend', 'weather_group', 'temp_diff']].head()

**Engineered Features:**
- **`time_of_day`** — groups hours into Night / Morning / Afternoon / Evening to capture daily patterns
- **`is_weekend`** — flags weekend days to separate leisure vs commuter behaviour
- **`weather_group`** — simplifies weather codes into readable business categories
- **`temp_diff`** — gap between actual and feels-like temperature; captures comfort perception

### 7.1 Demand by Time of Day

In [ ]:
df.groupby('time_of_day')['cnt'].mean().plot(
    kind='bar', color='steelblue', edgecolor='white', figsize=(8, 4))
plt.title('Average Bike Demand by Time of Day', fontweight='bold')
plt.ylabel('Average Count')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 8. EDA Conclusion

### 🔍 What the Data Tells Us

| Driver | Finding | Business Action |
|--------|---------|----------------|
| **Time** | Rush hours 8am and 5–6pm dominate weekday demand | Pre-position bikes at commuter hubs before peak hours |
| **Temperature** | Warm conditions (15–30°C) drive highest rentals | Run promotions during spring and fall |
| **Weather** | Clear weather = 3× demand vs rain | Have contingency plans for low-demand rainy days |
| **User Type** | Registered users = 80%+ of revenue | Prioritise retention — subscriptions and loyalty perks |
| **Season** | Fall highest, Winter lowest | Scale fleet and staffing seasonally |
| **Year-on-Year** | 2012 demand significantly higher than 2011 | System is growing — plan capacity expansion |

### 📌 Key Takeaway
Bike demand is primarily driven by **hour of day**, **temperature**, and **weather condition**. Registered commuters form the revenue core; casual users add weekend volume. The dataset is clean, structured, and ready for predictive modelling.

---

# `Model Building`

## 9. Encoding

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True)
print("Shape after encoding:", df_encoded.shape)
df_encoded.head()

## 10. Feature Selection

We drop columns that would **leak** the target (`cnt`):
- `casual` and `registered` — they sum to exactly `cnt`. Keeping them would give a trivial perfect score.
- `dteday` — raw datetime, not useful directly
- Derived columns (`demand_level`, `casual_ratio`, etc.)

This step is critical. **Target leakage** is the most common mistake in ML projects.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler

# Columns derived from target — must be removed to avoid leakage
leakage_cols = ['casual', 'registered', 'dteday',
                'demand_level', 'casual_ratio',
                'registered_ratio', 'usage_intensity', 'temp_bin']

drop_cols = [c for c in leakage_cols if c in df_encoded.columns]

model_df = df_encoded.drop(columns=drop_cols).copy()

X = model_df.drop(columns=['cnt'])
y = model_df['cnt']

print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]:,}")
print(f"Target   : cnt (hourly bike rentals)")

## 11. Train-Test Split

We split 80% for training and 20% for testing. `random_state=42` ensures reproducibility.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")

## 12. Scaling

We apply **StandardScaler** (mean=0, std=1) for consistency and best practice.

> **Important:** We fit the scaler on training data only and then transform both train and test. Fitting on test data would cause **data leakage**.

In [ ]:
scaler = StandardScaler()

X_train_sc = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns)
X_test_sc  = pd.DataFrame(scaler.transform(X_test),      columns=X.columns)

print("Scaling complete!")
print(f"Train mean (first feature): {X_train_sc.iloc[:,0].mean():.4f}  (should be ~0)")
print(f"Train std  (first feature): {X_train_sc.iloc[:,0].std():.4f}  (should be ~1)")

## 13. Evaluation Helper

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate(y_true, y_pred, label="Model"):
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    print(f"\n{'='*40}")
    print(f"  {label}")
    print(f"{'='*40}")
    print(f"  MAE  : {mae:.2f}")
    print(f"  RMSE : {rmse:.2f}")
    print(f"  R²   : {r2:.4f}")
    return mae, rmse, r2

---
## 14. Model Building — Decision Tree

Decision Tree Regressor is interpretable and handles non-linear relationships, but tends toward high variance.

---

In [ ]:
from sklearn.tree import DecisionTreeRegressor, plot_tree

### 14.1 Baseline Decision Tree

In [ ]:
dt_base = DecisionTreeRegressor(random_state=42)
dt_base.fit(X_train, y_train)
y_pred_dt_base = dt_base.predict(X_test)

evaluate(y_test, y_pred_dt_base, "Decision Tree — Baseline")

### 14.2 Hyperparameter Tuning — GridSearchCV

In [ ]:
param_grid_dt = {
    'max_depth'        : [3, 5, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 5],
}

grid_dt = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid_dt,
    cv=5,
    scoring='r2',
    n_jobs=-1,
)

grid_dt.fit(X_train, y_train)
print("Best Parameters:", grid_dt.best_params_)
print(f"Best CV R²     : {grid_dt.best_score_:.4f}")

### 14.3 Optimised Model Evaluation

In [ ]:
dt_best = grid_dt.best_estimator_
y_pred_dt = dt_best.predict(X_test)

dt_mae, dt_rmse, dt_r2 = evaluate(y_test, y_pred_dt, "Decision Tree — Optimised")

### 14.4 Actual vs Predicted — Decision Tree

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred_dt, alpha=0.3, color='steelblue', s=8)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.title('Decision Tree — Actual vs Predicted', fontweight='bold')
plt.xlabel('Actual Count')
plt.ylabel('Predicted Count')
plt.legend()
plt.tight_layout()
plt.show()

### 14.5 Tree Visualization (depth=3)

In [ ]:
plt.figure(figsize=(24, 10))
plot_tree(
    dt_best,
    max_depth=3,
    feature_names=X.columns,
    filled=True,
    rounded=True,
    fontsize=9,
)
plt.title("Decision Tree Visualization (max_depth=3)", fontweight='bold')
plt.tight_layout()
plt.show()

### 14.6 Feature Importances — Decision Tree

In [ ]:
fi_dt = pd.Series(dt_best.feature_importances_, index=X.columns)
fi_dt.nlargest(10).plot(kind='barh', color='steelblue', edgecolor='white', figsize=(9, 5))
plt.title("Top 10 Important Features — Decision Tree", fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## 15. Model Building — Random Forest

Random Forest builds many decision trees and averages their predictions, reducing variance and overfitting compared to a single tree.

---

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV

### 15.1 Baseline Random Forest

In [ ]:
rf_base = RandomForestRegressor(random_state=42, n_jobs=-1)
rf_base.fit(X_train, y_train)
y_pred_rf_base = rf_base.predict(X_test)

evaluate(y_test, y_pred_rf_base, "Random Forest — Baseline")

### 15.2 Hyperparameter Tuning — RandomizedSearchCV

In [ ]:
param_dist_rf = {
    'n_estimators'     : np.arange(100, 501, 50),
    'max_depth'        : [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10, 15],
    'min_samples_leaf' : [1, 2, 4, 6],
    'max_features'     : ['sqrt', 'log2', None],
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=param_dist_rf,
    n_iter=10,
    cv=3,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

rf_search.fit(X_train, y_train)
print("\nBest Parameters:", rf_search.best_params_)

### 15.3 Optimised Model Evaluation

In [ ]:
rf_best = rf_search.best_estimator_
y_pred_rf = rf_best.predict(X_test)

rf_mae, rf_rmse, rf_r2 = evaluate(y_test, y_pred_rf, "Random Forest — Optimised")

### 15.4 Overfitting Check

In [ ]:
y_train_pred_rf = rf_best.predict(X_train)

print("Overfitting Check — Random Forest")
print(f"  Train R² : {r2_score(y_train, y_train_pred_rf):.4f}")
print(f"  Test  R² : {rf_r2:.4f}")
print(f"  Gap      : {r2_score(y_train, y_train_pred_rf) - rf_r2:.4f}  (closer to 0 = better generalisation)")

### 15.5 Actual vs Predicted — Random Forest

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(y_test, y_pred_rf, alpha=0.3, color='#3498db', s=8)
plt.plot([y_test.min(), y_test.max()],
         [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
plt.title('Random Forest — Actual vs Predicted', fontweight='bold')
plt.xlabel('Actual Count')
plt.ylabel('Predicted Count')
plt.legend()
plt.tight_layout()
plt.show()

### 15.6 Feature Importances — Random Forest

In [ ]:
fi_rf = pd.Series(rf_best.feature_importances_, index=X.columns)\
           .sort_values(ascending=True).tail(15)

fi_rf.plot(kind='barh', color='#3498db', edgecolor='white', figsize=(9, 6))
plt.title("Top 15 Important Features — Random Forest", fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

---
## 16. Model Building — Gradient Boosting Regression

Gradient Boosting builds trees **sequentially** — each tree corrects the errors of the previous one. It is one of the most powerful algorithms for structured tabular data.

---

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error

print("Model libraries loaded!")

### 16.1 Baseline Gradient Boosting

In [ ]:
gbr_base = GradientBoostingRegressor(random_state=42)
gbr_base.fit(X_train_sc, y_train)
y_pred_gbr_base = gbr_base.predict(X_test_sc)

mae_base  = mean_absolute_error(y_test, y_pred_gbr_base)
rmse_base = np.sqrt(mean_squared_error(y_test, y_pred_gbr_base))
r2_base   = r2_score(y_test, y_pred_gbr_base)
mape_base = mean_absolute_percentage_error(y_test, y_pred_gbr_base) * 100

print("=" * 45)
print("  BASELINE MODEL — Default Parameters")
print("=" * 45)
print(f"  MAE   : {mae_base:.2f} bikes")
print(f"  RMSE  : {rmse_base:.2f} bikes")
print(f"  R²    : {r2_base:.4f}")
print(f"  MAPE  : {mape_base:.2f}%")
print("=" * 45)

### 16.2 Hyperparameter Tuning — GridSearchCV

| Parameter | What it controls |
|-----------|-----------------|
| `n_estimators` | Number of trees to build |
| `learning_rate` | Contribution of each tree — smaller = more conservative |
| `max_depth` | Depth of each tree — controls complexity |
| `min_samples_split` | Minimum samples needed to split a node |
| `subsample` | Fraction of training data used per tree — adds randomness |

`GridSearchCV` with `cv=3` tries every combination and picks the one with best cross-validated R² score.

In [ ]:
param_grid_gbr = {
    'n_estimators'     : [100, 200, 300],
    'learning_rate'    : [0.05, 0.1, 0.2],
    'max_depth'        : [3, 4, 5],
    'min_samples_split': [2, 5],
    'subsample'        : [0.8, 1.0],
}

gbr_grid = GridSearchCV(
    GradientBoostingRegressor(random_state=42),
    param_grid_gbr,
    cv=3,
    scoring='r2',
    n_jobs=-1,
    verbose=1,
)

print("Running GridSearchCV...")
gbr_grid.fit(X_train_sc, y_train)

print("\nBest Parameters:")
for p, v in gbr_grid.best_params_.items():
    print(f"  {p:22} : {v}")
print(f"\nBest CV R² : {gbr_grid.best_score_:.4f}")

### 16.3 Final Model Evaluation

In [ ]:
best_gbr     = gbr_grid.best_estimator_
y_pred_tuned = best_gbr.predict(X_test_sc)

mae_tuned  = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = np.sqrt(mean_squared_error(y_test, y_pred_tuned))
r2_tuned   = r2_score(y_test, y_pred_tuned)
mape_tuned = mean_absolute_percentage_error(y_test, y_pred_tuned) * 100

cv_scores = cross_val_score(best_gbr, X_train_sc, y_train, cv=5, scoring='r2')

print("=" * 52)
print("  GRADIENT BOOSTING — FINAL EVALUATION")
print("=" * 52)
print(f"  {'Metric':<18} {'Baseline':>10} {'Tuned':>10}")
print("-" * 52)
print(f"  {'MAE':<18} {mae_base:>10.2f} {mae_tuned:>10.2f}")
print(f"  {'RMSE':<18} {rmse_base:>10.2f} {rmse_tuned:>10.2f}")
print(f"  {'R²':<18} {r2_base:>10.4f} {r2_tuned:>10.4f}")
print(f"  {'MAPE (%)':<18} {mape_base:>10.2f} {mape_tuned:>10.2f}")
print("-" * 52)
print(f"  CV R² (5-fold)  : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print("=" * 52)

### 16.4 Evaluation Plots

Four plots to visually validate model performance:
1. **Actual vs Predicted** — points should cluster along the diagonal
2. **Residual Plot** — residuals should be randomly scattered around zero (no pattern)
3. **Feature Importances** — which features the model relied on most
4. **Residual Distribution** — should be approximately normal and centred at zero

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Actual vs Predicted
axes[0,0].scatter(y_test, y_pred_tuned, alpha=0.3, color='steelblue', s=8)
axes[0,0].plot([y_test.min(), y_test.max()],
               [y_test.min(), y_test.max()], 'r--', linewidth=2, label='Perfect Prediction')
axes[0,0].set_title('Actual vs Predicted Rentals', fontweight='bold')
axes[0,0].set_xlabel('Actual Count')
axes[0,0].set_ylabel('Predicted Count')
axes[0,0].legend()

# 2. Residual Plot
residuals = y_test - y_pred_tuned
axes[0,1].scatter(y_pred_tuned, residuals, alpha=0.3, color='orange', s=8)
axes[0,1].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0,1].set_title('Residual Plot', fontweight='bold')
axes[0,1].set_xlabel('Predicted Count')
axes[0,1].set_ylabel('Residual (Actual − Predicted)')

# 3. Top 15 Feature Importances
feat_imp = pd.Series(best_gbr.feature_importances_, index=X.columns)\
             .sort_values(ascending=True).tail(15)
feat_imp.plot(kind='barh', ax=axes[1,0], color='steelblue', edgecolor='white')
axes[1,0].set_title('Top 15 Feature Importances', fontweight='bold')
axes[1,0].set_xlabel('Importance Score')

# 4. Residual Distribution
sns.histplot(residuals, bins=50, kde=True, ax=axes[1,1], color='purple')
axes[1,1].axvline(x=0, color='red', linestyle='--', linewidth=2)
axes[1,1].set_title('Residual Distribution', fontweight='bold')
axes[1,1].set_xlabel('Residual Value')

plt.suptitle('Gradient Boosting Regression — Model Evaluation',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('gbr_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

### Model Summary

#### 📊 Metrics Explained

| Metric | Meaning |
|--------|---------|
| **MAE** | Average absolute error in number of bikes — directly interpretable |
| **RMSE** | Penalises large errors more — sensitive to outliers |
| **R²** | % of variance explained — 1.0 = perfect, 0 = random |
| **MAPE** | Average % error — easy to communicate to business |
| **CV R²** | Average R² over 5 folds — more reliable than single test score |

#### ✅ Interpretation
- High R² confirms the model captures demand patterns effectively
- Low MAE means predictions are within a small margin of actual rentals per hour
- Stable CV R² (low std) confirms the model generalises well to unseen data
- Feature importance reveals `hr`, `temp`, and `atemp` as top demand drivers — consistent with EDA

In [ ]:
print("=" * 52)
print("  GRADIENT BOOSTING — FINAL SUMMARY")
print("=" * 52)
print(f"  Algorithm      : Gradient Boosting Regressor")
print(f"  Train samples  : {len(X_train):,}")
print(f"  Test samples   : {len(X_test):,}")
print(f"  Features used  : {X.shape[1]}")
print()
print(f"  Best Parameters:")
for p, v in gbr_grid.best_params_.items():
    print(f"    {p:22} : {v}")
print()
print(f"  R² Score       : {r2_tuned:.4f}")
print(f"  MAE            : {mae_tuned:.2f} bikes/hour")
print(f"  RMSE           : {rmse_tuned:.2f} bikes/hour")
print(f"  MAPE           : {mape_tuned:.2f}%")
print(f"  CV R² (5-fold) : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print("=" * 52)

# `Model Comparison`

In [ ]:
# Collect metrics from all three models
model_names = ['Decision Tree', 'Random Forest', 'Gradient Boosting']
mae_all  = [dt_mae,  rf_mae,  mae_tuned]
rmse_all = [dt_rmse, rf_rmse, rmse_tuned]
r2_all   = [dt_r2,   rf_r2,   r2_tuned]

colors = ['#e74c3c', '#3498db', '#2ecc71']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, values, ylabel, title in [
    (axes[0], mae_all,  'MAE',      'MAE Comparison'),
    (axes[1], rmse_all, 'RMSE',     'RMSE Comparison'),
    (axes[2], r2_all,   'R² Score', 'R² Comparison'),
]:
    bars = ax.bar(model_names, values, color=colors, edgecolor='white', linewidth=0.8)

    # Highlight best bar
    best_idx = np.argmin(values) if ylabel != 'R² Score' else np.argmax(values)
    bars[best_idx].set_edgecolor('black')
    bars[best_idx].set_linewidth(2)

    for i, (bar, v) in enumerate(zip(bars, values)):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(values)*0.01,
                str(round(v, 2)), ha='center', va='bottom', fontsize=10)

    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.set_xticklabels(model_names, rotation=15, ha='right')
    ax.spines[['top', 'right']].set_visible(False)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Model Performance Comparison — All Three Models',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Ranked summary table
comparison_df = pd.DataFrame({
    'Model': model_names,
    'MAE'  : [round(v, 2) for v in mae_all],
    'RMSE' : [round(v, 2) for v in rmse_all],
    'R²'   : [round(v, 4) for v in r2_all],
})

comparison_df['MAE_rank']  = comparison_df['MAE'].rank().astype(int)
comparison_df['RMSE_rank'] = comparison_df['RMSE'].rank().astype(int)
comparison_df['R2_rank']   = comparison_df['R²'].rank(ascending=False).astype(int)
comparison_df['Avg_rank']  = comparison_df[['MAE_rank','RMSE_rank','R2_rank']].mean(axis=1)
comparison_df['Winner']    = comparison_df['Avg_rank'] == comparison_df['Avg_rank'].min()
comparison_df['Winner']    = comparison_df['Winner'].map({True: '✅', False: ''})

print(comparison_df.to_string(index=False))

---
### Analysis

| Model | Verdict |
|-------|---------|
| **Decision Tree** | High variance — lowest performance. Useful as an interpretable baseline only. |
| **Random Forest** | Strong ensemble — reduces variance significantly. Slight overfitting (Train R² > Test R²). |
| **Gradient Boosting** | **Best overall** — sequential error correction gives best generalisation. Stable CV score confirms reliability. |

#### 📌 Key Insights
- Decision Tree underperforms due to high variance
- Random Forest improves via ensemble averaging but shows mild overfit
- **Gradient Boosting performs best** — selected for deployment

---

# Model Deployment

In [ ]:
import joblib
from sklearn.pipeline import Pipeline

# Build pipeline: fitted scaler + best tuned GBR model
best_pipeline = Pipeline([
    ('scaler', scaler),     # already fitted on X_train
    ('model',  best_gbr),   # tuned Gradient Boosting
])

# Save all artefacts
joblib.dump(best_pipeline, 'best_model_pipeline.pkl')
joblib.dump(grid_dt,       'dt_model.pkl')
joblib.dump(rf_best,       'rf_model.pkl')
joblib.dump(best_gbr,      'gbr_model.pkl')
joblib.dump(scaler,        'scaler.pkl')

print("✅ All models and pipeline saved!")
print("   → best_model_pipeline.pkl  (GBR + scaler pipeline)")
print("   → dt_model.pkl             (Decision Tree GridSearchCV)")
print("   → rf_model.pkl             (Random Forest RandomizedSearchCV)")
print("   → gbr_model.pkl            (Gradient Boosting tuned)")
print("   → scaler.pkl               (fitted StandardScaler)")

In [ ]:
# Quick sanity check — reload and predict on one sample
pipeline_loaded = joblib.load('best_model_pipeline.pkl')

sample = X_test.iloc[[0]]
pred   = pipeline_loaded.predict(sample)[0]
actual = y_test.iloc[0]

print(f"Sample prediction : {pred:.0f} bikes")
print(f"Actual value      : {actual} bikes")
print(f"Error             : {abs(pred - actual):.0f} bikes")